In [55]:
import requests
import pandas as pd
import constants
import os

In [56]:
data = {
    "token":  constants.redcap_token,
    "content": "report",
    "format": "json",
    "csvDelimiter": "",
    "rawOrLabel": "raw",
    "rawOrLabelHeaders": "raw",
    "exportCheckboxLabel": "false",
    "returnFormat": "json"
}
data["report_id"] = "8161"

In [57]:
r = requests.post("https://redcap.rutgers.edu/api/", data=data)
r.status_code

200

In [58]:
demographics = pd.DataFrame(r.json())
demographics = demographics.replace("", pd.NA)
demographics = demographics.drop(columns=["redcap_repeat_instrument"]).groupby(["seqid", "redcap_repeat_instance"], as_index=False).first()
demographics = demographics.rename(columns={
    "physbatt_2f": "height_cm",
    "physbatt_2g": "mass_kg",
    "physbatt_2c": "waist_circumference_cm",
    "physbatt_2d": "hip_circumference_cm",
    "education_yrs": "education_years",
    "redcap_repeat_instance": "session",
    "seqid": "subject"
})
demographics

,subject,session,subjectid,test_date,neuroimaging_doadmin,fmri_notes,age_1,age,gender,race,ethnicity,education,education_years,height_cm,mass_kg,waist_circumference_cm,hip_circumference_cm
0,A001,1,AA001,2014-09-16,None,None,58,None,0,None,None,3,18,None,None,None,None
1,A002,1,AA002,2014-09-23,None,None,None,None,None,None,None,None,None,None,None,None,None
2,A003,1,AA003,2014-09-23,None,None,75,None,0,None,None,3,17,None,None,None,None
3,A003,2,AA_R003,2014-12-09,None,None,None,None,None,None,None,None,17,None,None,None,None
4,A003,3,None,None,None,None,None,None,None,None,None,None,17,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,L152,2,AA_R152_L,None,None,None,None,None,0,None,None,None,None,None,None,None,None
1618,L152,3,AA_3R152_L,2024-01-04,None,None,62,62,0,Black,1,1,13,162,71.2,86,109
1619,L259,1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1620,L270,1,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [59]:
def combine_cols(df, cols):
    result = []
    for index, row in df.iterrows():
        is_null = True
        for col in cols:
            if pd.notna(row[col]) and float(row[col]) > 0:
                result.append(row[col])
                is_null = False
                break
        if is_null:
            result.append(float("nan"))
    return result

def decode_education(education):
    if education == 0:
        return 12
    if education == 1:
        return 14
    if education == 2:
        return 16
    if education == 3:
        return 18
    if education == 4:
        return 20
    return float("nan")

def decode_ethnicity(ethnicity):
    if(ethnicity == "1"):
        return "Hispanic or Latino"
    elif(ethnicity == "0"):
        return "Not hispanic or latino"
    return pd.NA

def decode_gender(gender):
    if(gender == "1"):
        return "Male"
    elif(gender == "0"):
        return "Female"
    return pd.NA

In [60]:
age_cols = ["age_1", "age"]
edu_cols = ["education_years", "education"]
demographics.loc[:, "education"] = demographics["education"].apply(decode_education)
demographics.loc[:, "ethnicity"] = demographics["ethnicity"].apply(decode_ethnicity)
demographics.loc[:, "gender"] = demographics["gender"].apply(decode_gender)
demographics.loc[:, "education_years"] = combine_cols(demographics, edu_cols)
demographics.loc[:, "age"] = combine_cols(demographics, age_cols)
demographics = demographics.drop(columns=["age_1", "education"])

In [ ]:
prisma_subjects = []
prisma_sessions = []
for file in os.listdir(constants.dataset_path):
    if file.startswith("sub-"):
        sessions = [int(session[-1]) for session in os.listdir(os.path.join(constants.dataset_path, file))]
        prisma_sessions.extend(sessions)
        prisma_subjects.extend([file[4:]]*len(sessions))

prisma_subjects_sessions = pd.DataFrame({
    "subject": prisma_subjects,
    "session": prisma_sessions
})
prisma_subjects_sessions

,subject,session
0,C248,1
1,C327,1
2,C388,1
3,C002,2
4,A143,5
...,...,...
93,A044,3
94,C163,2
95,C177,1
96,C213,1


In [62]:
prisma_demographics = pd.merge(prisma_subjects_sessions, demographics, how="left")
prisma_demographics

,subject,session,subjectid,test_date,neuroimaging_doadmin,fmri_notes,age,gender,race,ethnicity,education_years,height_cm,mass_kg,waist_circumference_cm,hip_circumference_cm
0,C248,1,COV248,2023-11-29,None,None,69,Female,Black,<NA>,18,181,108.7,106,121
1,C327,1,COV327,2024-06-20,None,None,66,Female,Black,Hispanic or Latino,13,171,90.9,112,122
2,C388,1,COV388,2025-01-08,None,None,72,Female,Black,<NA>,15,166,82.5,108,114
3,C002,2,COV_R002,2023-09-07,None,None,65,Female,Black,Hispanic or Latino,12,153,82.9,112.5,110
4,A143,5,AA_4R143_L,2023-06-13,None,None,77,Female,Black,Hispanic or Latino,14,170.69,102.06,91.44,127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,A044,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94,C163,2,COV_R163,2025-02-21,None,None,81,Female,Black,<NA>,12,171,73.7,93,104
95,C177,1,COV177,2023-05-02,None,None,82,Female,Black,Hispanic or Latino,7,161,66.22,94.5,102
96,C213,1,COV213,2023-07-11,None,None,80,Female,Black,Hispanic or Latino,12,172.72,91.3,109.22,121.92


In [63]:
prisma_demographics.to_csv(os.path.join(constants.dataset_path, "participants.tsv"), sep="\t", index=False)